In [ ]:
import logging  #bilgi vermeyi sağlayan standart kütüphane bilgileri print ile vermektense logging içindeki farklı durumlara göre çıktı vermek daha doğru bir kullanımdır
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch #modelin sinir ağı bu kütüphane üzerinden çalışacak
import cv2
from transformers import AutoProcessor, AutoModelForMultimodalLM
#Processor image i modelin anlayabileceği sayısal verilere(çok boyutlu sayısal diziler) çevirir
#AutoModelForMultimodalLM → MiniCPM-V modelini yükler config dosyasına göre modeli yükler
#AutoProcessor            → Görsel + metni modele hazırlar

In [ ]:
#modelin ve processorun ortama yüklenmesi 
MODEL_ID = "openbmb/MiniCPM-V-4.6-BNB"

processor = AutoProcessor.from_pretrained(MODEL_ID) #Sadece görsel ve metni ileride nasıl hazırlayacağını bilen processor nesnesini oluşturuyor.
model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" #modelin nerede çalıştırılacağı otomatik belirlenecek (CPU veya GPU)
    )
model.eval() #eğitim değil de inference modunda çalıştırılacak. Katmanların çalışma davranışını inference'a uygun hale getirir.

In [ ]:
#modelin kurulumu durumu hakkında bilgilendirme
print("Model device:")
print(model.device)

print("Model class")
print(type(model))

total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_params}")

In [ ]:
#image in yüklenmesi ve sorunun belirlenmesi
IMAGE_PATH = Path("test_pictures\catt.jpg")

image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise ValueError(f"Image not found at {IMAGE_PATH}")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB) #renk kanallarını değiştiriyoruz

user_question = "Where is the cat in the image?"

question = """
Find the object requested by the user in the image.

Return only valid JSON in exactly this format:

[
    {
        "label": "object name",
        "box": [x1, y1, x2, y2]
    }
]

Bounding box coordinates must be integers from 0 to 1000.

Do not write any explanation before or after the JSON.

User request:
{user_question}
"""

In [ ]:
#chat modeli için inputu belirli bir konuşma yapısına getirilmesi
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_rgb
            },
            {
                "type": "text",
                "text": question
            }
        ]
    }
]

In [ ]:
#mesajın processor a verilmesi ve sonucunda inputun artık modelin anlayacağı bir formata dönüştürülmesi
inputs = processor.apply_chat_template(
    messages,
    tokenize=True, #girdileri tokenlara çevir
    add_generation_prompt=True, #özel tokenleri ekle
    return_dict=True, #girdileri dictionary formatında döndür
    return_tensors="pt", #çıktının tensor formatında olmasını sağla
)

inputs = inputs.to(model.device)

In [ ]:
#modelin cevap üretmesi
with torch.inference_mode(): #inference modunda olduğumuz için gradient hesaplarını kapatıyor
    outputs = model.generate(
        **inputs,
        max_new_tokens=100, #cevabın uzunluğu 100 token ile sınırlandırılıyor
    )

generated_tokens = outputs[:, inputs["input_ids"].shape[1]:] #model çıktı üretirken çıkışın başında inputu da eklediği için onu kesip sadece modelin ürettiği kısmı alıyoruz
#burada inputs["input_ids"] kısmı girdilerin dictionarysindeki girdi değerleirni alır
#.shade[1] ile de tensorün boyutunu gösterir
# : ile de tensör boyutu kadar olan kısımdaki tensörleri keser.

#çıktı oluşturulurken girdinin hemen arkasına yeni tokenler ekleniyor o yüzden cevabı oluşturuken promptu silmek gerekli

In [ ]:
# Modelin ürettiği cevabı token ID'lerinden string'e çevirme
response = processor.batch_decode(
    generated_tokens,
    skip_special_tokens=True,  # <eos>, <pad>, <assistant> gibi özel tokenları gösterme
)[0]  # Batch içindeki ilk cevabı al tek görsel tek soru olduğu için

print("Modelin ham cevabı:")
print(response)

In [ ]:
#çıktının JSON formatında olup olmadığını kontrol etme ve Python nesnesine dönüştürme
def parse_model_response(response):
        detections = json.loads(response)  # JSON stringini Python nesnesine dönüştür

        print(type(response))
        print(type(detections))
        print(detections)

        return detections

model
→ token ID tensoru
→ decode
→ string
→ json.loads()
→ Python list/dictionary

In [ ]:
def scale_box_to_pixels(box, width, height):
        #box = detections[0]["box"]
        #height, width = image_bgr.shape[:2]

        x1, y1, x2, y2 = box

        x1_px = int((x1 / 1000) * width)
        y1_px = int((y1 / 1000) * height)
        x2_px = int((x2 / 1000) * width)
        y2_px = int((y2 / 1000) * height)

        print("Model koordinatları:")
        print(x1, y1, x2, y2)

        print("Piksel koordinatları:")
        print(x1_px, y1_px, x2_px, y2_px)

        return x1_px, y1_px, x2_px, y2_px 

In [ ]:
def draw_bbox(image, box, label):
    height, width = image.shape[:2]

    x1_px, y1_px, x2_px, y2_px = scale_box_to_pixels(
        box,
        width,
        height
    )

    output_image = image.copy() #buradaki image a image_bgr verilecek

    cv2.rectangle(
        output_image,
        (x1_px, y1_px),
        (x2_px, y2_px),
        (0, 255, 0),
        2
    )

    cv2.putText(
        output_image,
        label,
        (x1_px, max(y1_px - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    return output_image

In [ ]:
#modelin oluşturduğu resmi rgb ye çevirme ve yazdırma
def show_bbox_coordinates(box, image_bgr, label):
    result_image = draw_bbox,(
        image_bgr,
        box,
        label
    )

    result_image_rgb = cv2.cvtColor(
        result_image,
        cv2.COLOR_BGR2RGB
    )

    plt.imshow(result_image_rgb)
    plt.axis("off")
    plt.show()